# Byte Pair Encoding (BPE)

Our simple tokenizer has a fundamental problem: it can't handle words it hasn't seen before. Byte Pair Encoding solves this by breaking words into subword units.

**Key idea:** Instead of whole words, BPE works with pieces of words. The word "unexpected" might become `["un", "expect", "ed"]`. This way, even new words can be represented using known pieces.

We'll use `tiktoken`, the tokenizer library used by GPT models.

In [1]:
import tiktoken
import importlib.metadata

print(f"tiktoken version: {importlib.metadata.version('tiktoken')}")

tiktoken version: 0.13.0


## Loading the GPT-2 Tokenizer

GPT-2's tokenizer has a vocabulary of about 50,000 tokens. These tokens were learned from a large text corpus using the BPE algorithm.

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
# Try encoding text with special tokens
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
    "of someunknownPlace."
)

# We need to explicitly allow special tokens
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Token IDs:\n{ids}")

Token IDs:
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [4]:
# Decode back to text
decoded = tokenizer.decode(ids)
print(f"\nDecoded:\n{decoded}")


Decoded:
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


## How BPE Handles Unknown Words

Notice "someunknownPlace" - a made-up word that definitely wasn't in GPT-2's training data. BPE breaks it into known subword pieces instead of failing.

In [5]:
# Let's see how an unknown word gets tokenized
word = "someunknownPlace"
ids = tokenizer.encode(word)

print(f"'{word}' becomes {len(ids)} tokens:")
for token_id in ids:
    token_text = tokenizer.decode([token_id])
    print(f"  {token_id} -> '{token_text}'")

'someunknownPlace' becomes 3 tokens:
  11246 -> 'some'
  34680 -> 'unknown'
  27271 -> 'Place'


## Special Tokens

By default, tiktoken raises an error if you try to encode special tokens like `<|endoftext|>`. You need to explicitly allow them.

In [6]:
# This will fail
try:
    tokenizer.encode("Hello <|endoftext|> world")
except ValueError as e:
    print(f"Error: {str(e)[:100]}...")

Error: Encountered text corresponding to disallowed special token '<|endoftext|>'.
If you want this text to...


In [7]:
# This works
ids = tokenizer.encode("Hello <|endoftext|> world", allowed_special={"<|endoftext|>"})
print(f"Token IDs: {ids}")

# The special token gets its own ID (50256)
print(f"\n<|endoftext|> token ID: {tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})}")

Token IDs: [15496, 220, 50256, 995]

<|endoftext|> token ID: [50256]


## Comparing Vocabulary Sizes

Our simple tokenizer had about 1,100 tokens (just the unique words in "The Verdict"). GPT-2's BPE tokenizer has 50,257 tokens - enough to represent any text efficiently.

In [8]:
print(f"GPT-2 vocabulary size: {tokenizer.n_vocab}")

GPT-2 vocabulary size: 50257


## Summary

BPE tokenization:
- Breaks text into subword units, not just whole words
- Can represent any input text, even made-up words
- Has a fixed vocabulary size (50,257 for GPT-2)
- More efficient than character-level tokenization

Next: We'll use this tokenizer to prepare training data for our model.